# 01 · AI Hub 데이터 구조 탐색

**목적**: 300GB를 통째로 받기 전에 파일 목록과 JSON 스키마를 확인한다.

전처리 스크립트는 실제 구조를 본 뒤에 작성한다. 추측으로 파서를 짜면 반드시 다시 써야 한다.

**런타임**: `수정 → 노트북 설정 → CPU` 로 설정하세요.
다운로드와 구조 확인에 GPU는 필요 없고, Colab Pro 컴퓨팅 유닛만 낭비됩니다.

**준비물**: AI Hub API Key (aihub.or.kr → 마이페이지 → API Key 발급 → 이메일 수신)

## 1. 디스크 확인

Colab 디스크는 100GB 안팎이다. 압축 해제에 원본의 2~3배가 필요하므로
한 번에 받을 파트 크기를 여기서 가늠한다.

In [ ]:
!df -h /content | tail -1
!free -g | head -2
!nproc

## 2. API Key 입력

`getpass`로 받으므로 노트북에 키가 저장되지 않는다. **키를 셀에 직접 쓰지 마세요** — 
GitHub에 푸시되면 노출됩니다.

In [ ]:
import os
from getpass import getpass

os.environ["AIHUB_KEY"] = getpass("AI Hub API Key: ").strip()
print("입력됨:", os.environ["AIHUB_KEY"][:8] + "...")

## 3. aihubshell 설치

In [ ]:
!curl -sS -o /usr/local/bin/aihubshell https://api.aihub.or.kr/api/aihubshell.do
!chmod +x /usr/local/bin/aihubshell
!aihubshell -help 2>&1 | head -30

## 4. 파일 목록 조회 ← **가장 중요한 셀**

여기 출력에 각 파일의 `filekey`와 용량이 나온다.
이걸 보고 어떤 파트를 어떤 순서로 나눠 받을지 정한다.

**출력 전체를 복사해서 저에게 주세요.** 그걸 보고 분할 계획과 전처리 코드를 짭니다.

In [ ]:
# 낙상사고 위험동작 영상-센서 쌍 데이터
DATASET_KEY = 71641

!aihubshell -mode l -datasetkey {DATASET_KEY} -aihubapikey "$AIHUB_KEY" 2>&1 | tee /content/filelist.txt

In [ ]:
# 목록이 길면 요약해서 보기
!wc -l /content/filelist.txt
print("=" * 70)
!head -60 /content/filelist.txt

## 5. 샘플 파트 하나만 다운로드

4번 출력에서 **가장 작은 Training 파트**의 filekey를 골라 아래에 넣으세요.
전체가 아니라 하나만 받아서 구조를 봅니다.

> `-filekey` 를 지정하지 않으면 300GB 전체가 받아져 디스크가 터집니다. 반드시 지정하세요.

In [ ]:
FILEKEY = ""  # ← 4번 출력을 보고 채우세요

assert FILEKEY, "4번 셀 출력에서 filekey를 확인해 채우세요"

!mkdir -p /content/raw
%cd /content/raw
!aihubshell -mode d -datasetkey {DATASET_KEY} -filekey {FILEKEY} -aihubapikey "$AIHUB_KEY"
%cd /content
!du -sh /content/raw

## 6. 압축 해제

AI Hub는 보통 분할 zip으로 내려온다.

In [ ]:
!find /content/raw -type f | head -30
print("=" * 70)
!find /content/raw -type f | wc -l

In [ ]:
# 분할 zip 병합 후 해제 (AI Hub 표준 방식)
%cd /content/raw
!apt-get install -qq zip unzip > /dev/null 2>&1

# .part 분할 파일이 있으면 병합
!for d in $(find . -name '*.part0' | xargs -n1 dirname | sort -u); do \
     echo "병합: $d"; \
     cat $d/*.part* > $d/merged.zip 2>/dev/null; \
   done

# zip 해제
!find . -name '*.zip' -exec sh -c 'unzip -qq -o "$1" -d "$(dirname "$1")"' _ {} \;
%cd /content
!du -sh /content/raw

## 7. 구조 파악 ← **이 출력도 저에게 주세요**

디렉토리 트리, 확장자 분포, JSON 스키마를 봅니다.

In [ ]:
from collections import Counter
from pathlib import Path

SRC = Path("/content/raw")

print("── 확장자 분포 " + "─" * 50)
exts = Counter(p.suffix.lower() for p in SRC.rglob("*") if p.is_file())
for e, c in exts.most_common(15):
    print(f"  {c:8,}  {e or '(확장자 없음)'}")

print("\n── 디렉토리 구조 (깊이 4) " + "─" * 40)
dirs = Counter()
for p in SRC.rglob("*"):
    if p.is_file():
        rel = p.relative_to(SRC)
        dirs[str(Path(*rel.parts[:4]).parent)] += 1
for d, c in dirs.most_common(25):
    print(f"  {c:8,}  {d}")

In [ ]:
import json

jsons = sorted(SRC.rglob("*.json"))
print(f"JSON 파일: {len(jsons):,}개\n")

# 샘플 3개의 전체 구조 출력
for jp in jsons[:3]:
    print("=" * 70)
    print(f"경로: {jp.relative_to(SRC)}")
    print("=" * 70)
    try:
        data = json.loads(jp.read_text(encoding="utf-8"))
        print(json.dumps(data, ensure_ascii=False, indent=2)[:3000])
    except Exception as e:
        print(f"읽기 실패: {e}")
    print()

In [ ]:
# 최상위 키 분포 — 스키마가 일관적인지 확인
key_counter = Counter()
for jp in jsons[:500]:
    try:
        d = json.loads(jp.read_text(encoding="utf-8"))
        if isinstance(d, dict):
            key_counter[tuple(sorted(d.keys()))] += 1
    except Exception:
        pass

print("최상위 키 조합 (상위 5종):")
for keys, cnt in key_counter.most_common(5):
    print(f"\n  [{cnt}건] {list(keys)}")

In [ ]:
# 이미지 샘플 확인 — 해상도와 실제 모습
import cv2
import matplotlib.pyplot as plt

imgs = [p for p in SRC.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
print(f"이미지: {len(imgs):,}개")

if imgs:
    res = Counter()
    for p in imgs[:200]:
        im = cv2.imread(str(p))
        if im is not None:
            res[im.shape[:2]] += 1
    print("\n해상도 분포:")
    for r, c in res.most_common(8):
        print(f"  {c:5,}  {r[1]}x{r[0]}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, p in zip(axes, imgs[:3]):
        im = cv2.imread(str(p))
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(p.name[:30], fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 8. 다음 단계

위 4·7번 출력을 복사해서 전달하면, 실제 스키마에 맞춘 전처리 스크립트를 작성합니다.

확인할 것:
- BBOX 좌표 형식 — `[x, y, w, h]` 인가 `[x1, y1, x2, y2]` 인가
- 낙상 클래스가 어느 키에 어떤 문자열로 들어있는가
- 라벨 JSON과 원천 이미지의 경로 대응 규칙
- 클립(시퀀스) 단위를 무엇으로 식별하는가 ← split 누수 방지에 필수